In [3]:
import cupy as cp
print(cp.cuda.is_available())
import cupyx

True


In [2]:
import os
print(os.environ.get('CUDA_PATH'))
print(os.environ.get('LD_LIBRARY_PATH'))
import cupy as cp
print(cp.cuda.is_available())

None
/usr/local/nvidia/lib:/usr/local/nvidia/lib64
True


In [5]:
import os
# Pick the directory that contains libcusparse.so.12
# The most likely one is from the pip-installed NVIDIA package:
os.environ['LD_LIBRARY_PATH'] = '/opt/conda/lib/python3.11/site-packages/nvidia/cusparse/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

In [1]:
import cupy as cp
from cupyx.scipy.sparse import csr_matrix

# Создаём массив с плавающей точкой
a = cp.array([[1, 0], [0, 1]], dtype=cp.float32)   # или cp.float64
sp = csr_matrix(a)
print(sp)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 2 stored elements and shape (2, 2)>
  Coords	Values
  (0, 0)	1.0
  (1, 1)	1.0


In [7]:
import site
import glob
# Ищем библиотеку в site-packages
for path in site.getsitepackages():
    libs = glob.glob(f"{path}/nvidia/cusparse/lib/libcusparse.so.12*")
    if libs:
        print(libs[0])   # покажет полный путь
        break

In [3]:
import os
conda_lib = os.path.expanduser('~/.conda/envs/film-recommendation/lib')
os.environ['LD_LIBRARY_PATH'] = f"{conda_lib}:" + os.environ.get('LD_LIBRARY_PATH', '')

In [2]:
import os
os.environ['CUDA_PATH'] = os.path.expanduser('~/.conda/envs/film-recommendation')
# Также убедимся, что LD_LIBRARY_PATH уже есть (можно продублировать)
os.environ['LD_LIBRARY_PATH'] = os.environ['CUDA_PATH'] + '/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

In [4]:
import os
conda_prefix = os.path.expanduser('~/.conda/envs/film-recommendation')
os.environ['CUDA_PATH'] = conda_prefix
os.environ['LD_LIBRARY_PATH'] = f"{conda_prefix}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')

In [1]:

import os, json
conda_prefix = os.environ['CONDA_PREFIX']
python_path = os.path.join(conda_prefix, 'bin/python')
cpath = (
    os.path.join(conda_prefix, 'lib/python3.11/site-packages/cupy/_core/include') + ':' +
    os.path.join(conda_prefix, 'lib/python3.11/site-packages/cupy/_core/include/cupy/_cuda/cuda-12') + ':' +
    os.path.join(conda_prefix, 'targets/x86_64-linux/include')
)
data = {
    'argv': [python_path, '-m', 'ipykernel_launcher', '-f', '{connection_file}'],
    'display_name': 'Python (film-recommendation)',
    'language': 'python',
    'env': {
        'LD_LIBRARY_PATH': os.path.join(conda_prefix, 'lib') + ':/usr/local/nvidia/lib:/usr/local/nvidia/lib64',
        'CUDA_PATH': conda_prefix,
        'CPATH': cpath
    }
}
os.makedirs(os.path.expanduser('~/.local/share/jupyter/kernels/film-recommendation'), exist_ok=True)
with open(os.path.expanduser('~/.local/share/jupyter/kernels/film-recommendation/kernel.json'), 'w') as f:
    json.dump(data, f, indent=2)


KeyError: 'CONDA_PREFIX'

In [2]:
import os
import sys

# Задаем путь к префиксу вашего conda-окружения
conda_prefix = os.environ.get("CONDA_PREFIX", f"{os.environ.get('HOME')}/.conda/envs/film-recommendation")

# Указываем CuPy, где искать заголовочные файлы CUDA
os.environ["CUPY_CUDA_COMPILE_WITH_PATH"] = conda_prefix
os.environ["CUDA_PATH"] = conda_prefix

# На всякий случай добавляем пути в системные переменные для компилятора
if os.path.exists(f"{conda_prefix}/include"):
    os.environ["CPATH"] = f"{conda_prefix}/include"

In [1]:
import os
import sys
import shutil
import subprocess

print("=== 1. Окружение Python ===")
print(f"Python executable: {sys.executable}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', 'Не задан')}")

print("\n=== 2. Переменные CUDA в системе ===")
for var in ['CUDA_PATH', 'CUDA_HOME', 'CUDA_ROOT', 'CPATH', 'CUPY_CUDA_COMPILE_WITH_PATH']:
    print(f"{var}: {os.environ.get(var, 'Не задан')}")

print("\n=== 3. Проверка компилятора NVCC ===")
nvcc_path = shutil.which("nvcc")
print(f"nvcc в PATH: {nvcc_path}")
if nvcc_path:
    try:
        res = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
        print(res.stdout.strip())
    except Exception as e:
        print(f"Ошибка вызова nvcc: {e}")

print("\n=== 4. Поиск файла cuda_fp16.h ===")
# Ищем заголовочный файл в типичных местах
possible_paths = [
    os.environ.get("CONDA_PREFIX", ""),
    "/usr/local/cuda",
    "/usr/include",
]
found = False
for p in possible_paths:
    if not p: continue
    for root, dirs, files in os.walk(p):
        if "cuda_fp16.h" in files:
            print(f"НАЙДЕН: {os.path.join(root, 'cuda_fp16.h')}")
            found = True
            break
if not found:
    print("Файл cuda_fp16.h НЕ найден в стандартных путях.")

print("\n=== 5. Версия CuPy ===")
try:
    import cupy
    print(f"CuPy version: {cupy.__version__}")
    print(f"CuPy CUDA version: {cupy.cuda.runtime.runtimeGetVersion()}")
except Exception as e:
    print(f"Ошибка импорта CuPy: {e}")

=== 1. Окружение Python ===
Python executable: /home/jovyan/.conda/envs/film-recommendation/bin/python
CONDA_PREFIX: Не задан

=== 2. Переменные CUDA в системе ===
CUDA_PATH: Не задан
CUDA_HOME: Не задан
CUDA_ROOT: Не задан
CPATH: Не задан
CUPY_CUDA_COMPILE_WITH_PATH: Не задан

=== 3. Проверка компилятора NVCC ===
nvcc в PATH: None

=== 4. Поиск файла cuda_fp16.h ===
Файл cuda_fp16.h НЕ найден в стандартных путях.

=== 5. Версия CuPy ===
CuPy version: 13.6.0
CuPy CUDA version: 12090


In [3]:
import os
import sys

# Явно задаем путь к вашему окружению, где теперь лежат бинарники CUDA
env_path = "/home/jovyan/.conda/envs/film-recommendation"

os.environ["CUDA_PATH"] = env_path
os.environ["CUDA_HOME"] = env_path
os.environ["CUPY_CUDA_COMPILE_WITH_PATH"] = env_path

# Прокидываем путь к nvcc и заголовочным файлам
os.environ["PATH"] = f"{env_path}/bin:" + os.environ.get("PATH", "")
os.environ["CPATH"] = f"{env_path}/include:" + os.environ.get("CPATH", "")

print("Переменные окружения CUDA успешно настроены!")

Переменные окружения CUDA успешно настроены!


In [5]:
import os
import sys

# Находим путь к заголовочным файлам вашего активного conda-окружения
conda_include_dir = f"{sys.prefix}/include"

# Передаем этот путь в компилятор CuPy через системные флаги
os.environ["NVCC_APPEND_FLAGS"] = f"-I{conda_include_dir}"

print(f"Путь для поиска заголовков CUDA успешно добавлен: {conda_include_dir}")

Путь для поиска заголовков CUDA успешно добавлен: /home/jovyan/.conda/envs/film-recommendation/include


In [8]:
import json
import os

# Путь конкретно к твоему используемому ядру film-recommendation
kernel_json_path = os.path.expanduser("~/.local/share/jupyter/kernels/film-recommendation/kernel.json")

if os.path.exists(kernel_json_path):
    with open(kernel_json_path, "r") as f:
        data = json.load(f)
    
    # Жестко зашиваем пути к CUDA внутри окружения прямо в ядро
    conda_env = "/home/jovyan/.conda/envs/film-recommendation"
    
    data["env"] = {
        "CUDA_PATH": conda_env,
        "CUDA_HOME": conda_env,
        "CPATH": f"{conda_env}/include",
        "NVCC_APPEND_FLAGS": f"-I{conda_env}/include",
        "LD_LIBRARY_PATH": f"{conda_env}/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64"
    }
    
    with open(kernel_json_path, "w") as f:
        json.dump(data, f, indent=2)
        
    print("✅ Отлично! kernel.json для ядра film-recommendation успешно обновлен.")
    print("👉 Теперь ЖЕСТКО ПЕРЕЗАПУСТИ ЯДРО (Kernel -> Restart Kernel) и запускай обучение!")
else:
    print(f"❌ Хм, файл по пути {kernel_json_path} всё ещё не найден.")

✅ Отлично! kernel.json для ядра film-recommendation успешно обновлен.
👉 Теперь ЖЕСТКО ПЕРЕЗАПУСТИ ЯДРО (Kernel -> Restart Kernel) и запускай обучение!


In [7]:
 !jupyter kernelspec list

Available kernels:
  film-rec-env           /home/jovyan/.local/share/jupyter/kernels/film-rec-env
  film-recommendation    /home/jovyan/.local/share/jupyter/kernels/film-recommendation
  python3                /home/jovyan/.conda/envs/film-recommendation/share/jupyter/kernels/python3


In [1]:
!rm -rf ~/.cupy/kernel_cache

In [1]:
import cupy as cp
print(cp.__version__)

13.4.1


In [4]:
import fastrlock
print(fastrlock.__version__)

0.8.3


In [3]:
import numpy as np
import pandas as pd
import cupy as cp
print(np.__version__)
print(pd.__version__)
print(cp.__version__)

2.4.6
3.0.3
13.6.0


In [2]:
import cupy as cp
import cupyx.scipy.sparse as sp

# Создаем фейковую матрицу, явно используя float
A = sp.csr_matrix(cp.array([[1.0, 2.0], [3.0, 4.0]]))

# Делаем срез (именно тут вызывается JIT-компиляция ядра)
print("Срез работает:\n", A[0:1].todense())

Срез работает:
 [[1. 2.]]


In [2]:
!pip list | grep implicit   # если используете ! в Jupyter

In [1]:
import implicit

# Check if the GPU extension is available
print("GPU Support Available:", implicit.gpu.HAS_CUDA)

ModuleNotFoundError: No module named 'implicit'

In [4]:
import sys
print(sys.executable)

/home/jovyan/.conda/envs/film-recommendation/bin/python


In [2]:
import cupy
cupy.show_config()

OS                           : Linux-7.0.12-201.fc44.x86_64-x86_64-with-glibc2.35
Python Version               : 3.11.15
CuPy Version                 : 13.6.0
CuPy Platform                : NVIDIA CUDA
NumPy Version                : 2.4.6
SciPy Version                : 1.16.3
Cython Build Version         : 3.1.3
Cython Runtime Version       : 3.3.0a2.dev0
CUDA Root                    : None
nvcc PATH                    : None
CUDA Build Version           : 11080
CUDA Driver Version          : 13020
CUDA Runtime Version         : 11080 (linked to CuPy) / 11080 (locally installed)
CUDA Extra Include Dirs      : []
cuBLAS Version               : (available)
cuFFT Version                : 10900
cuRAND Version               : 10300
cuSOLVER Version             : (11, 4, 1)
cuSPARSE Version             : (available)
NVRTC Version                : (11, 8)
Thrust Version               : 200800
CUB Build Version            : 200800
Jitify Build Version         : <unknown>
cuDNN Build Version   

In [1]:
import sys, os, subprocess, platform, pkg_resources

print("="*60)
print("DIAGNOSTIC REPORT")
print("="*60)

print("\n--- System ---")
print(f"Platform: {platform.platform()}")
print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Conda env: {os.environ.get('CONDA_DEFAULT_ENV', 'NOT SET')}")

print("\n--- Conda packages (filtered) ---")
try:
    result = subprocess.run(['conda', 'list'], capture_output=True, text=True, check=False)
    for line in result.stdout.splitlines():
        if any(p in line.lower() for p in ['implicit', 'cupy', 'cudatoolkit', 'cuda', 'nccl']):
            print(line)
except Exception as e:
    print(f"Error: {e}")

print("\n--- pip packages (implicit & cupy) ---")
try:
    for dist in pkg_resources.working_set:
        if dist.project_name.lower() in ['implicit', 'cupy', 'cupy-cuda']:
            print(dist)
except Exception as e:
    print(f"Error: {e}")

print("\n--- CuPy ---")
try:
    import cupy
    print(f"Version: {cupy.__version__}")
    print(f"CUDA available: {cupy.cuda.is_available()}")
    if cupy.cuda.is_available():
        print(f"Device ID: {cupy.cuda.runtime.getDevice()}")
    print("Config:")
    cupy.show_config()
except Exception as e:
    print(f"Import error: {e}")

print("\n--- Implicit ---")
try:
    import implicit
    print(f"Version: {implicit.__version__}")
    print(f"GPU HAS_CUDA: {implicit.gpu.HAS_CUDA}")
    # test import of GPU ALS
    try:
        from implicit.gpu import alternating_least_squares
        print("GPU ALS import: OK")
    except Exception as e:
        print(f"GPU ALS import error: {e}")
except Exception as e:
    print(f"Import error: {e}")

print("\n--- CUDA Environment Variables ---")
for var in ['CUDA_PATH', 'CUDA_HOME', 'CUDA_VISIBLE_DEVICES', 'LD_LIBRARY_PATH', 'PATH', 'CONDA_PREFIX']:
    print(f"{var}: {os.environ.get(var, 'NOT SET')}")

print("\n--- nvcc ---")
try:
    nvcc = subprocess.run(['nvcc', '--version'], capture_output=True, text=True, timeout=5)
    if nvcc.returncode == 0:
        print(nvcc.stdout.strip())
    else:
        print("nvcc not found (returncode != 0)")
except Exception as e:
    print(f"nvcc error: {e}")

print("\n--- Loading CUDA libraries (ctypes) ---")
if platform.system() == 'Windows':
    libs = ['cudart64_11.dll', 'cublas64_11.dll', 'cudart64_12.dll', 'cublas64_12.dll']
else:
    libs = ['libcudart.so', 'libcublas.so', 'libcudart.so.11.0', 'libcublas.so.11']
for lib in libs:
    try:
        import ctypes
        ctypes.CDLL(lib)
        print(f"{lib} -> loaded OK")
    except Exception as e:
        print(f"{lib} -> failed: {e}")

print("\n--- CuPy device array test ---")
try:
    import cupy as cp
    a = cp.array([1,2,3])
    print(f"Array created on device: {a.device}")
except Exception as e:
    print(f"Array creation error: {e}")

print("\n" + "="*60)
print("DIAGNOSTIC COMPLETE")
print("="*60)

DIAGNOSTIC REPORT

--- System ---
Platform: Linux-7.0.12-201.fc44.x86_64-x86_64-with-glibc2.35
Python: 3.11.15 | packaged by conda-forge | (main, Jun 11 2026, 03:34:02) [GCC 14.3.0]
Executable: /home/jovyan/.conda/envs/film-recommendation/bin/python
Conda env: NOT SET

--- Conda packages (filtered) ---


/tmp/ipykernel_30757/3963767562.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import sys, os, subprocess, platform, pkg_resources


nvidia-cuda-cupti-cu12    12.4.127                 pypi_0    pypi
nvidia-cuda-nvrtc-cu12    12.4.127                 pypi_0    pypi
nvidia-cuda-runtime-cu12  12.4.127                 pypi_0    pypi
nvidia-nccl-cu12          2.21.5                   pypi_0    pypi

--- pip packages (implicit & cupy) ---
cupy 13.6.0
implicit 0.7.2

--- CuPy ---
Version: 13.6.0
CUDA available: True
Device ID: 0
Config:
OS                           : Linux-7.0.12-201.fc44.x86_64-x86_64-with-glibc2.35
Python Version               : 3.11.15
CuPy Version                 : 13.6.0
CuPy Platform                : NVIDIA CUDA
NumPy Version                : 2.4.6
SciPy Version                : 1.16.3
Cython Build Version         : 3.1.3
Cython Runtime Version       : 3.3.0a2.dev0
CUDA Root                    : None
nvcc PATH                    : None
CUDA Build Version           : 11080
CUDA Driver Version          : 13020
CUDA Runtime Version         : 11080 (linked to CuPy) / 11080 (locally installed)
CUDA Extra 